In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["GEMINI_API_KEY"] = os.getenv("gemini")

In [2]:
### Summarization Middleware

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.tools import tool

In [ ]:
agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.5-flash-lite",
            trigger=("messages", 10),
            keep=("messages", 5),
        )
    ],
)

In [ ]:
### run with the thread id
config = {"configurable": {"thread_id": "test-1"}}

In [ ]:
questions = [
    "What is the capital of France?",
    "What is the population of France?",
    "What is the currency of France?",
    "What is the official language of France?",
    "What is the largest city in France?",
    "What is the national anthem of France?",
    "What is the national holiday of France?",
    "What is the national emblem of France?",
]

In [ ]:
for q in questions:
    response = agent.invoke({"messages": [HumanMessage(content=q)]}, config=config)
    print(f"Q: {q}\nA: {response}\n")
    print(f"Messages:{len(response['messages'])}\n")

Q: What is the capital of France?
A: {'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nThe user is asking for general factual information about France, including its capital, population, currency, official language, largest city, and national anthem.\n\n## SUMMARY\n\n- Identified the capital of France as Paris.\n- Provided the population of France as approximately 68.4 million (as of January 2024).\n- Identified the currency of France as the euro (€).\n- Identified the official language of France as French.\n- Identified the largest city in France as Paris.\n- Identified the national anthem of France as "La Marseillaise" (written in 1792 by Claude Joseph Rouget de Lisle, adopted in 1795).\n\n## ARTIFACTS\n\nNone\n\n## NEXT STEPS\n\nAnswer any additional factual questions the user may have about France or other topics.', additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='929e509e-dbfd-40e6-a63d-cf08cbd3f51

In [ ]:
@tool
def search_hotels(location: str) -> str:
    """
    Search for hotels in a given location and return the results.
    """
    # For demonstration purposes, we'll return a static response.
    return f"""Hotels in {location}
        1. Grand Hotel 5 star, $350/night,spa, pool, free breakfast
        2. Budget Inn 3 star, $100/night, free parking, free breakfast
        """

In [ ]:
agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.5-flash-lite",
            trigger=("tokens", 400),
            keep=("tokens", 200),
        )
    ],
    tools=[search_hotels],
)

In [ ]:
agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.5-flash-lite",
            trigger=("fraction", 0.005),
            keep=("fraction", 0.002),
        )
    ],
    tools=[search_hotels],
)  # 0.5% = 650 tokens, 0.2% = 260 tokens

In [ ]:
config = {"configurable": {"thread_id": "test-2"}}


def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # Rough estimate: 1 token ~ 4 characters

In [ ]:
cities = ["New York", "Los Angeles", "Chicago", "Houston", "Phoenix"]
for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Search for hotels in {city}")]},
        config=config,
    )
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages\n")

New York: ~160 tokens, 4 messages

Los Angeles: ~323 tokens, 8 messages

Chicago: ~483 tokens, 12 messages

Houston: ~643 tokens, 16 messages

Phoenix: ~803 tokens, 20 messages



Human In the Loop


In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

In [ ]:
def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID:{email_id}"


def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [ ]:
agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {"allowed_decisions": ["approve", "reject", "edit"]},
                "read_email_tool": False,
            }
        )
    ],
)

In [ ]:
config = {"configurable": {"thread_id": "test-approve"}}
config = {"configurable": {"thread_id": "test-reject"}}
# Step 1:Request
result = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Send an email to john@gmail.com with subject hello and body How are you?"
            )
        ]
    },
    config=config,
)

In [ ]:
config = {"configurable": {"thread_id": "test-reject"}}

In [78]:
result

{'messages': [HumanMessage(content='Send an email to john@gmail.com with subject hello and body How are you?', additional_kwargs={}, response_metadata={}, id='616d9236-5236-4a4b-b2fa-f56b7d8ebb1d'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"recipient": "john@gmail.com", "subject": "hello", "body": "How are you?"}'}, '__gemini_function_call_thought_signatures__': {'call_1704864': 'EmAKXgFpFH0TZ6ES+DRYGW6QZFMatymP6EAUTccPWDbZOO50qRUbwP65zW58kjjN/s0iLme8lEg9Ua0vdoeKSZDvgrB4FT4q0LfBen9EvAzxoetfPZcr+vs4rXs1xYhXuLA='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a13f88-2a1a-7112-80f9-10b628c932b4-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@gmail.com', 'subject': 'hello', 'body': 'How are you?'}, 'id': 'call_1704864', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 

In [79]:
from langgraph.types import Command

In [ ]:
if "__interrupt__" in result:
    print("Paused! Approving the action...")
    result = agent.invoke(
        Command(resume={"decisions": [{"type": "reject"}]}), config=config
    )
    print("Resumed! Result:", result["messages"][-1].content)

Paused! Approving the action...
Resumed! Result: [{'type': 'text', 'text': 'The email sending was rejected, so I cannot send the email to john@gmail.com. Let me know if you need help with anything else!', 'extras': {'signature': 'EmAKXgFpFH0TsBTGXNhabD30HMWkedH4n/u+Oe/jQLF9yJNJ4RzFNmtuBobcvftvgyWPW7qmB9o+tFDWf+aHqhZpIuVfmiierwglCET1oANYNoXK/3EDotFJ2zEbyfwXbis='}}]


In [95]:
config = {"configurable": {"thread_id": "test-edit"}}
result = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Send an email to john@gmail.com with subject hello and body How are you?"
            )
        ]
    },
    config=config,
)

In [96]:
result

{'messages': [HumanMessage(content='Send an email to john@gmail.com with subject hello and body How are you?', additional_kwargs={}, response_metadata={}, id='5465d7ab-a9df-4370-804c-57a9256cf449'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"subject": "hello", "body": "How are you?", "recipient": "john@gmail.com"}'}, '__gemini_function_call_thought_signatures__': {'call_1390880': 'EmAKXgFpFH0TdxVRdLnYP4la1Vxa6H6Osi9HjjjpE6SysZaLp4kY7nes5VT/GqNtnII2EOSgvjm8JuLRkkaBtmgzxVmxBRyxj++7K730OEmpKMfCmsZh3N9DqAoSU61TYgc='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a13f89-6df2-78d3-b47f-acbc4970e7eb-0', tool_calls=[{'name': 'send_email_tool', 'args': {'subject': 'hello', 'body': 'How are you?', 'recipient': 'john@gmail.com'}, 'id': 'call_1390880', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 

In [97]:
# Step 2: Edit the action
if "__interrupt__" in result:
    print("Paused! Editing the action...")
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",
                            "args": {
                                "recipient": "abcd@gmail.com",
                                "subject": "AI",
                                "body": "generate summay on AI",
                            },
                        },
                    }
                ]
            }
        ),
        config=config,
    )
    print("Resumed! Result:", result["messages"][-1].content)

Paused! Editing the action...
Resumed! Result: [{'type': 'text', 'text': 'I have sent the email to john@gmail.com with the subject "hello" and the body "How are you?".', 'extras': {'signature': 'EmAKXgFpFH0Te4XYgBpnPWHcsk59UbT8ur/DBHG3oWm1eQZupoh1usERqWypbYsbj6YFSEmUNc983ybHocGH18Dremc76JVh8AFyyr8B1kTm4E5AqsmoJRSs6z9xCathPv0='}}]


In [98]:
result

{'messages': [HumanMessage(content='Send an email to john@gmail.com with subject hello and body How are you?', additional_kwargs={}, response_metadata={}, id='5465d7ab-a9df-4370-804c-57a9256cf449'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"subject": "hello", "body": "How are you?", "recipient": "john@gmail.com"}'}, '__gemini_function_call_thought_signatures__': {'call_1390880': 'EmAKXgFpFH0TdxVRdLnYP4la1Vxa6H6Osi9HjjjpE6SysZaLp4kY7nes5VT/GqNtnII2EOSgvjm8JuLRkkaBtmgzxVmxBRyxj++7K730OEmpKMfCmsZh3N9DqAoSU61TYgc='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a13f89-6df2-78d3-b47f-acbc4970e7eb-0', tool_calls=[{'name': 'send_email_tool', 'args': {'subject': 'hello', 'body': 'How are you?', 'recipient': 'john@gmail.com'}, 'id': 'call_1390880', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 